# DTE Saturation Attack: Large-Scale Numerical Verification\n\n## Objective\nExhaustive numerical validation of DTE Theorems 1-4 across high dimensions,\nlarge sample sizes, and edge cases.\n\n**DTE Framework**: https://github.com/chepin-ai/DTE-Project

In [ ]:
# Install DTE package\n!pip install numpy scipy matplotlib -q\n!git clone https://github.com/chepin-ai/DTE-Project.git\nimport sys\nsys.path.insert(0, 'DTE-Project/python')

In [ ]:
from dte.core import DTECoreEngine, DTETriple\nfrom dte.states import StateGenerator\nimport numpy as np\nimport time\nimport json\nimport matplotlib.pyplot as plt\nfrom concurrent.futures import ProcessPoolExecutor

## Configuration

In [ ]:
CONFIG = {\n    "dims": [(2,2), (2,3), (3,3), (4,4), (5,5), (6,6), (7,7), (8,8)],\n    "n_pure_samples": 10000,      # Theorem 1\n    "n_mixed_samples": 5000,       # Theorem 2\n    "n_werner_points": 100,       # Werner state scan\n    "tolerance": 1e-10,\n}

## Theorem 1: G = O (Exact Equality)

In [ ]:
def verify_theorem1(da, db, n_samples):\n    engine = DTECoreEngine(da, db)\n    gen = StateGenerator()\n    max_diff = 0.0\n    for seed in range(n_samples):\n        rho = gen.random_pure_state(da * db, seed=seed)\n        g = engine.G(rho)\n        o = engine.O(rho)\n        diff = abs(g - o)\n        max_diff = max(max_diff, diff)\n    return {\n        "dims": f"{da}x{db}",\n        "max_diff": float(max_diff),\n        "passed": bool(max_diff < 1e-12)\n    }\n\nresults_t1 = []\nfor da, db in CONFIG["dims"]:\n    r = verify_theorem1(da, db, CONFIG["n_pure_samples"])\n    results_t1.append(r)\n    print(f"Theorem 1 {r['dims']}: max|G-O|={r['max_diff']:.2e} | {'PASS' if r['passed'] else 'FAIL'}")

## Theorem 2: I >= c(d) * G^2

In [ ]:
def verify_theorem2(da, db, n_samples):\n    engine = DTECoreEngine(da, db)\n    gen = StateGenerator()\n    d = min(da, db)\n    c_d = 8.0 * np.log2(d) / ((d - 1) ** 2) if d > 1 else 8.0\n    min_ratio = float('inf')\n    for seed in range(n_samples):\n        rho = gen.random_pure_state(da * db, seed=seed) if seed % 2 == 0 else gen.random_mixed_state(da * db, seed=seed)\n        t = engine.triple(rho)\n        if t.G > 1e-10:\n            ratio = t.I / (t.G ** 2)\n            min_ratio = min(min_ratio, ratio)\n    return {\n        "dims": f"{da}x{db}",\n        "min_ratio": float(min_ratio),\n        "c_d": float(c_d),\n        "passed": bool(min_ratio >= c_d - 1e-6)\n    }\n\nresults_t2 = []\nfor da, db in CONFIG["dims"]:\n    r = verify_theorem2(da, db, CONFIG["n_mixed_samples"])\n    results_t2.append(r)\n    print(f"Theorem 2 {r['dims']}: min(I/G^2)={r['min_ratio']:.4f} >= c(d)={r['c_d']:.4f} | {'PASS' if r['passed'] else 'FAIL'}")

## Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n\n# Theorem 1 precision\nax1 = axes[0]\ndims_t1 = [r['dims'] for r in results_t1]\ndiffs = [r['max_diff'] for r in results_t1]\ncolors = ['#2ecc71' if r['passed'] else '#e74c3c' for r in results_t1]\nax1.bar(dims_t1, diffs, color=colors, alpha=0.7, edgecolor='black')\nax1.set_yscale('log')\nax1.set_ylabel('max |G - O|')\nax1.set_title('Theorem 1: G = O Precision')\nax1.axhline(y=1e-12, color='red', linestyle='--', alpha=0.5, label='Tolerance')\nax1.legend()\n\n# Theorem 2 lower bound\nax2 = axes[1]\ndims_t2 = [r['dims'] for r in results_t2]\nratios = [r['min_ratio'] for r in results_t2]\nc_ds = [r['c_d'] for r in results_t2]\nx = np.arange(len(dims_t2))\nwidth = 0.35\nax2.bar(x - width/2, ratios, width, label='min(I/G^2)', color='#3498db', alpha=0.7)\nax2.bar(x + width/2, c_ds, width, label='c(d) theory', color='#e74c3c', alpha=0.7)\nax2.set_xticks(x)\nax2.set_xticklabels(dims_t2)\nax2.set_ylabel('Ratio')\nax2.set_title('Theorem 2: I >= c(d) * G^2')\nax2.legend()\n\nplt.tight_layout()\nplt.savefig('dte_theorems_validation.png', dpi=150, bbox_inches='tight')\nplt.show()

## Save Results

In [ ]:
final_results = {\n    "config": CONFIG,\n    "theorem1": results_t1,\n    "theorem2": results_t2,\n    "summary": {\n        "theorem1_all_pass": all(r['passed'] for r in results_t1),\n        "theorem2_all_pass": all(r['passed'] for r in results_t2),\n        "total_samples": len(CONFIG['dims']) * CONFIG['n_pure_samples'] + len(CONFIG['dims']) * CONFIG['n_mixed_samples']\n    }\n}\n\nwith open('dte_saturation_results.json', 'w') as f:\n    json.dump(final_results, f, indent=2)\n\nprint(f"\nResults saved to dte_saturation_results.json")\nprint(f"Total samples: {final_results['summary']['total_samples']}")\nprint(f"Theorem 1: {'ALL PASS' if final_results['summary']['theorem1_all_pass'] else 'SOME FAILED'}")\nprint(f"Theorem 2: {'ALL PASS' if final_results['summary']['theorem2_all_pass'] else 'SOME FAILED'}")